In [2]:
!qvm -S

/bin/bash: qvm: command not found


In [4]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile
from qiskit.primitives import BackendEstimatorV2
from qiskit_rigetti import RigettiQCSProvider

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 12                 # Increased to 12 to support 180 nodes at k=2
k = 2                           # Quadratic compression

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        # This loop makes it dynamically work for any k
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
# High shots reduce statistical noise for the parameter shift gradients
estimator.options.default_shots = 10000

experiment_result = []


best_params = np.load("pce_optimal_theta_adam_reps4qbt12Continue2004.npy")

print("\nAdam Optimization Complete!")
np.save("pce_optimal_theta_adam_reps4qbt12Continue2004.npy", best_params)

# ==========================================
# 7. FINAL EVALUATION ON PYQUIL QVM
# ==========================================
print("\nPreparing to run final optimized circuit on PyQuil QVM...")

# Bind the trained parameters to the ansatz to create a fixed circuit
bound_circuit = ansatz.assign_parameters(best_params)

# Initialize PyQuil QVM via the Qiskit-Rigetti provider
provider = RigettiQCSProvider()
qvm_backend = provider.get_simulator(f"{num_qubits}q-qvm")

# Wrap the PyQuil backend with Qiskit's V2 Estimator
estimator_qvm = BackendEstimatorV2(backend=qvm_backend)
estimator_qvm.options.default_shots = 10000

print("Submitting job to QVM...")
job_qvm = estimator_qvm.run([
    (bound_circuit, observables_x),
    (bound_circuit, observables_y),
    (bound_circuit, observables_z)
])

results_qvm = job_qvm.result()

E_x_qvm = results_qvm[0].data.evs
E_y_qvm = results_qvm[1].data.evs
E_z_qvm = results_qvm[2].data.evs

# These are the final expectation values calculated by Rigetti's QVM
E_final_qvm = np.concatenate([E_x_qvm, E_y_qvm, E_z_qvm])

print("Expectation values from PyQuil QVM successfully retrieved!")

# ==========================================
# 8. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

# Decode the bits directly from the QVM expectations (E_final_qvm) instead of Aer
cur_bits = []
for i in range(num_nodes):
    if E_final_qvm[i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the initial base cut based on QVM
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"\nInitial Cut Size from QVM before Local Search: {best_cut}")

# Single-bit swap search
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] # Flip 1 to 0 or 0 to 1
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    # Keep the swap if it improves the cut
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size (Post QVM Evaluation):", best_cut)

ModuleNotFoundError: No module named 'qiskit.extensions'

In [5]:
pip install qiskit.extensions

ERROR: Could not find a version that satisfies the requirement qiskit.extensions (from versions: none)
ERROR: No matching distribution found for qiskit.extensions
Note: you may need to restart the kernel to use updated packages.


In [12]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile
from qiskit.primitives import BackendEstimatorV2


# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 12                 # Increased to 12 to support 180 nodes at k=2
k = 2                           # Quadratic compression

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        # This loop makes it dynamically work for any k
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
# High shots reduce statistical noise for the parameter shift gradients
estimator.options.default_shots = 10000

experiment_result = []

Loading dataset...


/tmp/ipykernel_195921/327494188.py:61: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)


In [18]:
from pytket.extensions.qiskit import qiskit_to_tk
from pytket.extensions.pyquil import tk_to_pyquil
from pyquil.paulis import sI, sX, sY, sZ
from pytket.passes import AutoRebase
from pyquil.api import WavefunctionSimulator
import numpy as np
from pytket.extensions.qiskit import qiskit_to_tk
from pytket.extensions.pyquil import tk_to_pyquil
from pytket.passes import AutoRebase
from pytket.circuit import OpType
from pyquil.paulis import sI, sX, sY, sZ
from pyquil.api import WavefunctionSimulator
import numpy as np

# ==========================================
# 8. TRANSLATE TO NATIVE PYQUIL VIA PYTKET
# ==========================================
print("\nTranslating Qiskit circuit to PyQuil via PyTKET...")

# 1. Bind the trained parameters to your Qiskit ansatz
best_params_loaded = np.load("pce_optimal_theta_adam_reps4qbt12Continue2004.npy")
bound_circuit = ansatz.assign_parameters(best_params_loaded)

# 2. Convert Qiskit -> TKET -> PyQuil Program
tket_circ = qiskit_to_tk(bound_circuit)


rebase = AutoRebase({OpType.CX, OpType.Rx, OpType.Rz})
rebase.apply(tket_circ)

pyquil_prog = tk_to_pyquil(tket_circ)

# 3. Translate Qiskit SparsePauliOp observables to PyQuil PauliTerms
def qiskit_to_pyquil_pauli(qiskit_op):
    """
    Converts a Qiskit SparsePauliOp to a PyQuil PauliTerm.
    Note: Qiskit strings are read right-to-left for qubit 0 -> n.
    """
    pauli_str = qiskit_op.paulis.to_labels()[0]
    term = sI()  # Start with the Identity multiplier
    
    # Enumerate backwards to match Qiskit's endianness
    for qubit_idx, char in enumerate(reversed(pauli_str)):
        if char == 'X':
            term *= sX(qubit_idx)
        elif char == 'Y':
            term *= sY(qubit_idx)
        elif char == 'Z':
            term *= sZ(qubit_idx)
    return term

print("Translating observables...")
pyquil_obs_x = [qiskit_to_pyquil_pauli(op) for op in observables_x]
pyquil_obs_y = [qiskit_to_pyquil_pauli(op) for op in observables_y]
pyquil_obs_z = [qiskit_to_pyquil_pauli(op) for op in observables_z]

# ==========================================
# 9. EVALUATE ON PYQUIL QVM & DECODE
# ==========================================
print("Evaluating exact expectation values on PyQuil QVM WavefunctionSimulator...")

wfn_sim = WavefunctionSimulator()

# Calculate expectation values by passing the entire list of PauliTerms directly!
# This natively returns a numpy array, which is much faster than looping.
E_x_qvm = np.real(wfn_sim.expectation(pyquil_prog, pauli_terms=pyquil_obs_x))
E_y_qvm = np.real(wfn_sim.expectation(pyquil_prog, pauli_terms=pyquil_obs_y))
E_z_qvm = np.real(wfn_sim.expectation(pyquil_prog, pauli_terms=pyquil_obs_z))

E_final_qvm = np.concatenate([E_x_qvm, E_y_qvm, E_z_qvm])
print("QVM Evaluation Complete!\n")

def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

# Decode the bits directly from the QVM expectations
cur_bits = []
for i in range(num_nodes):
    if E_final_qvm[i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the initial base cut based on QVM
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"Initial Cut Size from PyQuil QVM before Local Search: {best_cut}")

# Single-bit swap search
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] 
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size (Post QVM):", best_cut)


Translating Qiskit circuit to PyQuil via PyTKET...
Translating observables...
Evaluating exact expectation values on PyQuil QVM WavefunctionSimulator...
QVM Evaluation Complete!

Initial Cut Size from PyQuil QVM before Local Search: 6260.286774176787
Final Optimized Cut Size (Post QVM): 6528.332178117385


In [3]:
pip install pytket-pyquil

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 8.9 MB/s eta 0:00:00a 0:00:01m
  Attempting uninstall: qcs-sdk-python
    Found existing installation: qcs-sdk-python 0.21.18
    Uninstalling qcs-sdk-python-0.21.18:
      Successfully uninstalled qcs-sdk-python-0.21.180/2 [qcs-sdk-python]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pytket-pyquil]0m [pytket-pyquil]
Note: you may need to restart the kernel to use updated packages.


In [19]:
import networkx as nx
import numpy as np
from pyquil import get_qc

print("Connecting to Ankaa-3 to fetch live topology...")
qc = get_qc("Ankaa-3")

# 1. Get the physical layout of the chip as a NetworkX graph
# Nodes are physical qubits, edges are microwave couplers
topology = qc.quantum_processor.qubit_topology()
isa = qc.quantum_processor.to_compiler_isa()

def get_edge_fidelity(q1, q2, isa):
    """
    Helper to extract the 2-qubit fidelity from the ISA.
    If the API doesn't expose it directly today, we default to 1.0 (perfect)
    so the algorithm falls back to purely finding the tightest cluster.
    """
    # In a production environment, you pull the CZ or CPhaee fidelity here.
    # We will mock it as a random high fidelity for the sake of the search structure,
    # but the topology graph guarantees we only pick physically connected qubits.
    return np.random.uniform(0.90, 0.99) 

def find_best_12_qubit_cluster(topology, isa, target_size=12):
    """Greedy search to find the most tightly connected, high-fidelity cluster."""
    best_cluster = []
    best_cluster_score = 0.0
    
    # Check every possible starting physical qubit
    for start_node in topology.nodes():
        cluster = set([start_node])
        neighbors = set(topology.neighbors(start_node))
        score = 0.0
        
        # Grow the cluster until it reaches 12 qubits
        while len(cluster) < target_size and neighbors:
            # Score all available neighbors based on their connection to the cluster
            best_neighbor = None
            best_edge_score = -1
            
            for n in neighbors:
                # Find which qubit in the cluster it connects to
                for c in cluster:
                    if topology.has_edge(c, n):
                        fid = get_edge_fidelity(c, n, isa)
                        if fid > best_edge_score:
                            best_edge_score = fid
                            best_neighbor = n
            
            if best_neighbor is not None:
                cluster.add(best_neighbor)
                neighbors.remove(best_neighbor)
                score += best_edge_score
                # Add the new neighbor's neighbors to our frontier
                for nn in topology.neighbors(best_neighbor):
                    if nn not in cluster:
                        neighbors.add(nn)
            else:
                break # Reached a dead end on the chip
                
        # If we successfully formed a 12-qubit cluster, check if it's the best one yet
        if len(cluster) == target_size:
            if score > best_cluster_score:
                best_cluster_score = score
                best_cluster = list(cluster)
                
    return best_cluster

# 2. Execute the search
print("Scanning chip for the optimal 12-qubit neighborhood...")
best_physical_qubits = find_best_12_qubit_cluster(topology, isa, target_size=num_qubits)
print(f"Optimal Physical Qubits Selected: {best_physical_qubits}")

# 3. Create the mapping dictionary (Logical 0-11 -> Physical X-Y)
qubit_mapping = {logical: physical for logical, physical in enumerate(best_physical_qubits)}
print("Mapping Dictionary created:", qubit_mapping)

Connecting to Ankaa-3 to fetch live topology...
Scanning chip for the optimal 12-qubit neighborhood...
Optimal Physical Qubits Selected: [37, 38, 44, 45, 51, 52, 53, 58, 59, 60, 30, 31]
Mapping Dictionary created: {0: 37, 1: 38, 2: 44, 3: 45, 4: 51, 5: 52, 6: 53, 7: 58, 8: 59, 9: 60, 10: 30, 11: 31}


In [21]:
from pyquil import Program, get_qc
from pyquil.gates import H, RX, MEASURE
from pyquil.quil import address_qubits
import numpy as np

# ==========================================
# 9. APPLY MAPPING & EXECUTE ON ANKAA-3
# ==========================================
print("Preparing circuit for Rigetti Ankaa-3...")

# Your chosen high-fidelity patch
qubit_mapping = {
    0: 37, 1: 38, 2: 44, 3: 45, 4: 51, 5: 52, 
    6: 53, 7: 58, 8: 59, 9: 60, 10: 30, 11: 31
}

# 1. Map the logical PyTKET program to your physical Ankaa-3 qubits
mapped_base_prog = address_qubits(pyquil_prog, qubit_mapping)

qc = get_qc("Ankaa-3")
shots = 5000

def run_mapped_basis(mapped_prog, basis, mapping, qc, shots):
    """Appends basis transformations, compiles for Ankaa-3, and executes."""
    # Enforce our mapping starting position
    prog = Program(f'PRAGMA INITIAL_REWIRE "NAIVE"')
    prog += mapped_prog
    
    # Declare classical register to hold our 12 bits
    ro = prog.declare("ro", "BIT", len(mapping))
    
    # 1. APPLY ALL GATES FIRST
    # Apply basis rotations to the physical qubits
    for logical_q, physical_q in mapping.items():
        if basis == 'X':
            prog += H(physical_q)
        elif basis == 'Y':
            prog += RX(np.pi/2, physical_q)
            
    # 2. APPLY ALL MEASUREMENTS LAST
    # Store physical qubit measurement in the corresponding logical index
    for logical_q, physical_q in mapping.items():
        prog += MEASURE(physical_q, ro[logical_q])
        
    prog.wrap_in_numshots_loop(shots)
    
    print(f"Compiling {basis}-basis circuit...")
    exe = qc.compile(prog)
    
    print(f"Executing {basis}-basis circuit on Ankaa-3...")
    result = qc.run(exe)
    
    return result.get_register_map().get("ro")

# 2. Run the 3 global tasks on the physical hardware
res_x = run_mapped_basis(mapped_base_prog, 'X', qubit_mapping, qc, shots)
res_y = run_mapped_basis(mapped_base_prog, 'Y', qubit_mapping, qc, shots)
res_z = run_mapped_basis(mapped_base_prog, 'Z', qubit_mapping, qc, shots)

# ==========================================
# 10. DECODE BITSTRINGS & CALCULATE CUT
# ==========================================
def get_expectations(bitstrings, qiskit_observables, shots):
    """Translates raw 0s and 1s into expectation values for our specific observables."""
    spins = 1 - 2 * bitstrings 
    evs = []
    
    for obs in qiskit_observables:
        pauli_str = list(reversed(obs.paulis.to_labels()[0]))
        obs_val = np.ones(shots)
        
        for q, char in enumerate(pauli_str):
            if char != 'I':
                obs_val *= spins[:, q] 
                
        evs.append(np.mean(obs_val))
    return np.array(evs)

print("\nCalculating expectations from Ankaa-3 telemetry...")
# Because we mapped the readouts back to logical indices (0-11), 
# our original observables list works perfectly here!
E_x_qpu = get_expectations(res_x, observables_x, shots)
E_y_qpu = get_expectations(res_y, observables_y, shots)
E_z_qpu = get_expectations(res_z, observables_z, shots)

E_final_qpu = np.concatenate([E_x_qpu, E_y_qpu, E_z_qpu])
print("Ankaa-3 QPU Evaluation Complete!\n")

def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if E_final_qpu[i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"Initial Cut Size from physical Ankaa-3 before Local Search: {best_cut}")

for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] 
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size (Post Ankaa-3 execution):", best_cut)

Preparing circuit for Rigetti Ankaa-3...
Compiling X-basis circuit...
Executing X-basis circuit on Ankaa-3...
Compiling Y-basis circuit...
Executing Y-basis circuit on Ankaa-3...
Compiling Z-basis circuit...
Executing Z-basis circuit on Ankaa-3...

Calculating expectations from Ankaa-3 telemetry...
Ankaa-3 QPU Evaluation Complete!

Initial Cut Size from physical Ankaa-3 before Local Search: 3670.8557633620667
Final Optimized Cut Size (Post Ankaa-3 execution): 6086.775601382021


In [22]:
# ==========================================
# 7. DEEP DECODING & BIT-SWAP POST-PROCESSING
# ==========================================

# 1. Extract initial bitstring from the Quantum Circuit's Expectation Values
cur_bits = np.zeros(num_nodes, dtype=int)
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits[i] = 1
    else:
        cur_bits[i] = 0

# Helper to calculate the full absolute cut size once
def calculate_full_cut(graph, bits):
    cut = 0
    for u, v, data in graph.edges(data=True):
        if bits[u] != bits[v]:
            cut += data.get('weight', 1.0)
    return cut

current_cut = calculate_full_cut(graph, cur_bits)
print(f"\nInitial Quantum Cut (Before Local Search): {current_cut}")

# 2. Iterative 1-Bit and 2-Bit Swap Search
search_iteration = 0
improvement_found = True

while improvement_found:
    improvement_found = False
    search_iteration += 1
    
    # --- PHASE 1: 1-BIT SWAP SEARCH ---
    # We sweep through all nodes. If flipping a node improves the cut, we keep it.
    for u in graph.nodes():
        delta = 0
        # Calculate exactly how the cut changes if we flip node `u`
        for v in graph.neighbors(u):
            w = graph[u][v].get('weight', 1.0)
            if cur_bits[u] == cur_bits[v]:
                delta += w  # They were in the same partition. Flipping u cuts this edge.
            else:
                delta -= w  # They were in different partitions. Flipping u un-cuts it.
                
        # If flipping strictly improves the cut, apply the flip
        if delta > 0:
            cur_bits[u] = 1 - cur_bits[u]
            current_cut += delta
            improvement_found = True

    # --- PHASE 2: 2-BIT SWAP SEARCH ---
    # If 1-bit swaps get stuck in a local minimum, we try flipping pairs of connected nodes
    if not improvement_found:
        for u, v in graph.edges():
            delta = 0
            
            # Change in cut for neighbors of u (excluding v)
            for n_u in graph.neighbors(u):
                if n_u == v: continue
                w = graph[u][n_u].get('weight', 1.0)
                if cur_bits[u] == cur_bits[n_u]: delta += w
                else: delta -= w
                    
            # Change in cut for neighbors of v (excluding u)
            for n_v in graph.neighbors(v):
                if n_v == u: continue
                w = graph[v][n_v].get('weight', 1.0)
                if cur_bits[v] == cur_bits[n_v]: delta += w
                else: delta -= w
                    
            if delta > 0:
                cur_bits[u] = 1 - cur_bits[u]
                cur_bits[v] = 1 - cur_bits[v]
                current_cut += delta
                improvement_found = True
                break # Break to restart the 1-bit sweep with the new landscape

print(f"Post-processing complete after {search_iteration} sweeps.")
print(f"Final Deep-Optimized Cut Size: {current_cut}")

IndexError: list index out of range

In [23]:
native_quil = qc.compiler.quil_to_native_quil(pyquil_prog)
print(native_quil)  # Count the 2-qubit gates — each one decoheres your state

RX(pi/2) 29
RZ(0.8459373174668969) 29
RX(-pi/2) 29
RZ(-1.8224358516634038) 29
RX(pi/2) 30
RZ(0.0366610873505315) 30
RX(-pi/2) 30
RZ(0.3600316186307828) 30
ISWAP 29 30
RZ((-3*pi)/2) 29
RX(pi/2) 29
RZ((3*pi)/2) 29
ISWAP 29 30
RZ(-pi) 31
RX(pi/2) 31
RZ(1.5995695088258717) 31
RX(-pi/2) 31
RZ(2.5310892979230446) 31
RX(pi/2) 38
RZ(0.9434857895017591) 38
RX(-pi/2) 38
RZ(1.5326158088409936) 38
ISWAP 31 38
RZ((-3*pi)/2) 31
RX(pi/2) 31
RZ((3*pi)/2) 31
ISWAP 31 38
RZ(pi) 9
RX(pi/2) 9
RZ(3.128432734927167) 9
RX(-pi/2) 9
RZ(1.6527893638825206) 9
RZ(pi) 16
RX(pi/2) 16
RZ(0.9220990094142252) 16
RX(-pi/2) 16
RZ(-2.9174539301484304) 16
ISWAP 9 16
RZ((-3*pi)/2) 9
RX(pi/2) 9
RZ((3*pi)/2) 9
ISWAP 9 16
RZ(pi) 22
RX(pi/2) 22
RZ(2.923205315619126) 22
RX(-pi/2) 22
RZ(-1.4872269582048996) 22
RZ(pi) 23
RX(pi/2) 23
RZ(3.031926812088836) 23
RX(-pi/2) 23
RZ(-2.4305982803872515) 23
ISWAP 23 22
RZ((-3*pi)/2) 23
RX(pi/2) 23
RZ((3*pi)/2) 23
ISWAP 23 22
RZ(-pi) 38
RX(pi/2) 38
RZ((3*pi)/2) 38
RZ(-pi) 44
RX(pi/2) 44
RZ(2

In [25]:
from pyquil.quil import Program

# native_quil is already generated
# native_quil = qc.compiler.quil_to_native_quil(pyquil_prog)

two_qubit_count = sum(
    1 for inst in native_quil
    if hasattr(inst, "qubits") and len(inst.qubits) == 1
)

print("Number of 2-qubit gates:", two_qubit_count)

Number of 2-qubit gates: 450


In [26]:
from qiskit_aer import AerSimulator
from qiskit.primitives import BackendEstimatorV2
from qiskit import transpile
from qiskit.quantum_info import SparsePauliOp

# 1. Setup Backend
# Use AerSimulator to mimic a clean environment first
backend = AerSimulator()
estimator = BackendEstimatorV2(backend=backend)

# 2. Transpile for the backend
# Use the EXACT same basis gates you targeted for Ankaa-3
target_basis = ['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x']
qc_transpiled = transpile(bound_circuit, backend, basis_gates=target_basis)

# 3. Combine your observables (X, Y, and Z sets)
# You have lists of observables_x, observables_y, observables_z
# We combine them into a single structure for the Estimator
all_observables = observables_x + observables_y + observables_z

# 4. Run using EstimatorV2 (Pubs approach)
# We bind the circuit to the job
print("Executing via Qiskit BackendEstimatorV2...")
job = estimator.run([(qc_transpiled, all_observables)])
result = job.result()

# 5. Process results
# Result is a primitive job result. Access the first pub.
pub_result = result[0]
evs = pub_result.data.evs # These are your expectation values

# Split them back out to match your logic
E_final_qiskit = evs
print("Qiskit Estimator Execution Complete!")

Executing via Qiskit BackendEstimatorV2...


/opt/conda/envs/python3/lib/python3.11/site-packages/qiskit/compiler/transpiler.py:269: UserWarning: Providing `coupling_map` and/or `basis_gates` along with `backend` is not recommended, as this will invalidate the backend's gate durations and error rates.
  pm = generate_preset_pass_manager(


Qiskit Estimator Execution Complete!


In [27]:
from qiskit_aer.primitives import EstimatorV2 as AerEstimator

# 1. Setup the Noiseless Shot-based Simulator
# We use 5000 shots to match your Ankaa-3 run
shots = 5000
aer_estimator = AerEstimator()
aer_estimator.options.default_shots = shots

# 2. Combine all observables into one list for a single job
all_observables = observables_x + observables_y + observables_z

# 3. Create the Pub (Primitive Unified Bloc)
# (circuit, observables, parameter_values)
pub = (ansatz, all_observables, best_params_loaded)

print(f"Running shot-based simulation ({shots} shots)...")
job = aer_estimator.run([pub])
result = job.result()[0]

# 4. Extract expectation values
# Result.data.evs contains the expectation values for each observable
E_final_aer = result.data.evs
print("Shot-based Evaluation Complete!")

# 5. Decode and Calculate Cut
cur_bits_aer = [1 if val >= 0 else 0 for val in E_final_aer]

def get_cut(bits):
    p = [set(), set()]
    for i, b in enumerate(bits):
        p[0].add(i) if b > 0 else p[1].add(i)
    return calc_cut_size(graph, p[0], p[1])

aer_cut = get_cut(cur_bits_aer)
print(f"Initial Cut Size (Aer Shots): {aer_cut}")

Running shot-based simulation (5000 shots)...
Shot-based Evaluation Complete!
Initial Cut Size (Aer Shots): 6260.286774176787


In [28]:
from pyquil import Program, get_qc
from pyquil.gates import H, RX, MEASURE
from pyquil.quil import address_qubits
import numpy as np

# ==========================================
# 9. APPLY MAPPING & EXECUTE ON ANKAA-3
# ==========================================
print("Preparing circuit for Rigetti Ankaa-3...")

# Your chosen high-fidelity patch
qubit_mapping = {
    0: 37, 1: 38, 2: 44, 3: 45, 4: 51, 5: 52, 
    6: 53, 7: 58, 8: 59, 9: 60, 10: 30, 11: 31
}

# 1. Map the logical PyTKET program to your physical Ankaa-3 qubits
mapped_base_prog = address_qubits(pyquil_prog, qubit_mapping)

qc = get_qc("Ankaa-3")
shots = 10000

def run_mapped_basis(mapped_prog, basis, mapping, qc, shots):
    """Appends basis transformations, compiles for Ankaa-3, and executes."""
    # Enforce our mapping starting position
    prog = Program(f'PRAGMA INITIAL_REWIRE "NAIVE"')
    prog += mapped_prog
    
    # Declare classical register to hold our 12 bits
    ro = prog.declare("ro", "BIT", len(mapping))
    
    # 1. APPLY ALL GATES FIRST
    # Apply basis rotations to the physical qubits
    for logical_q, physical_q in mapping.items():
        if basis == 'X':
            prog += H(physical_q)
        elif basis == 'Y':
            prog += RX(np.pi/2, physical_q)
            
    # 2. APPLY ALL MEASUREMENTS LAST
    # Store physical qubit measurement in the corresponding logical index
    for logical_q, physical_q in mapping.items():
        prog += MEASURE(physical_q, ro[logical_q])
        
    prog.wrap_in_numshots_loop(shots)
    
    print(f"Compiling {basis}-basis circuit...")
    exe = qc.compile(prog)
    
    print(f"Executing {basis}-basis circuit on Ankaa-3...")
    result = qc.run(exe)
    
    return result.get_register_map().get("ro")

# 2. Run the 3 global tasks on the physical hardware
res_x = run_mapped_basis(mapped_base_prog, 'X', qubit_mapping, qc, shots)
res_y = run_mapped_basis(mapped_base_prog, 'Y', qubit_mapping, qc, shots)
res_z = run_mapped_basis(mapped_base_prog, 'Z', qubit_mapping, qc, shots)

# ==========================================
# 10. DECODE BITSTRINGS & CALCULATE CUT
# ==========================================
def get_expectations(bitstrings, qiskit_observables, shots):
    """Translates raw 0s and 1s into expectation values for our specific observables."""
    spins = 1 - 2 * bitstrings 
    evs = []
    
    for obs in qiskit_observables:
        pauli_str = list(reversed(obs.paulis.to_labels()[0]))
        obs_val = np.ones(shots)
        
        for q, char in enumerate(pauli_str):
            if char != 'I':
                obs_val *= spins[:, q] 
                
        evs.append(np.mean(obs_val))
    return np.array(evs)

print("\nCalculating expectations from Ankaa-3 telemetry...")
# Because we mapped the readouts back to logical indices (0-11), 
# our original observables list works perfectly here!
E_x_qpu = get_expectations(res_x, observables_x, shots)
E_y_qpu = get_expectations(res_y, observables_y, shots)
E_z_qpu = get_expectations(res_z, observables_z, shots)

E_final_qpu = np.concatenate([E_x_qpu, E_y_qpu, E_z_qpu])
print("Ankaa-3 QPU Evaluation Complete!\n")

def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if E_final_qpu[i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"Initial Cut Size from physical Ankaa-3 before Local Search: {best_cut}")

for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] 
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size (Post Ankaa-3 execution):", best_cut)

Preparing circuit for Rigetti Ankaa-3...
Compiling X-basis circuit...
Executing X-basis circuit on Ankaa-3...
Compiling Y-basis circuit...
Executing Y-basis circuit on Ankaa-3...
Compiling Z-basis circuit...
Executing Z-basis circuit on Ankaa-3...

Calculating expectations from Ankaa-3 telemetry...
Ankaa-3 QPU Evaluation Complete!

Initial Cut Size from physical Ankaa-3 before Local Search: 2978.791882030141
Final Optimized Cut Size (Post Ankaa-3 execution): 6038.215961979625


In [34]:
from pytket.extensions.qiskit import qiskit_to_tk
from pytket.extensions.pyquil import tk_to_pyquil
from pyquil.paulis import sI, sX, sY, sZ
from pytket.passes import AutoRebase
from pyquil.api import WavefunctionSimulator
import numpy as np
from pytket.extensions.qiskit import qiskit_to_tk
from pytket.extensions.pyquil import tk_to_pyquil
from pytket.passes import AutoRebase
from pytket.circuit import OpType
from pyquil.paulis import sI, sX, sY, sZ
from pyquil.api import WavefunctionSimulator
import numpy as np

# ==========================================
# 8. TRANSLATE TO NATIVE PYQUIL VIA PYTKET
# ==========================================
print("\nTranslating Qiskit circuit to PyQuil via PyTKET...")

# 1. Bind the trained parameters to your Qiskit ansatz




df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 12                 # Increased to 12 to support 180 nodes at k=2
k = 2                           # Quadratic compression

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        # This loop makes it dynamically work for any k
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=3)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])








best_params_loaded = np.load("pce_optimal_theta_adam_reps4qbt12Continue2008.npy")
bound_circuit = ansatz.assign_parameters(best_params_loaded)

# 2. Convert Qiskit -> TKET -> PyQuil Program
tket_circ = qiskit_to_tk(bound_circuit)


rebase = AutoRebase({OpType.CX, OpType.Rx, OpType.Rz})
rebase.apply(tket_circ)

pyquil_prog = tk_to_pyquil(tket_circ)

# 3. Translate Qiskit SparsePauliOp observables to PyQuil PauliTerms
def qiskit_to_pyquil_pauli(qiskit_op):
    """
    Converts a Qiskit SparsePauliOp to a PyQuil PauliTerm.
    Note: Qiskit strings are read right-to-left for qubit 0 -> n.
    """
    pauli_str = qiskit_op.paulis.to_labels()[0]
    term = sI()  # Start with the Identity multiplier
    
    # Enumerate backwards to match Qiskit's endianness
    for qubit_idx, char in enumerate(reversed(pauli_str)):
        if char == 'X':
            term *= sX(qubit_idx)
        elif char == 'Y':
            term *= sY(qubit_idx)
        elif char == 'Z':
            term *= sZ(qubit_idx)
    return term

print("Translating observables...")
pyquil_obs_x = [qiskit_to_pyquil_pauli(op) for op in observables_x]
pyquil_obs_y = [qiskit_to_pyquil_pauli(op) for op in observables_y]
pyquil_obs_z = [qiskit_to_pyquil_pauli(op) for op in observables_z]

# ==========================================
# 9. EVALUATE ON PYQUIL QVM & DECODE
# ==========================================
print("Evaluating exact expectation values on PyQuil QVM WavefunctionSimulator...")

wfn_sim = WavefunctionSimulator()

# Calculate expectation values by passing the entire list of PauliTerms directly!
# This natively returns a numpy array, which is much faster than looping.
E_x_qvm = np.real(wfn_sim.expectation(pyquil_prog, pauli_terms=pyquil_obs_x))
E_y_qvm = np.real(wfn_sim.expectation(pyquil_prog, pauli_terms=pyquil_obs_y))
E_z_qvm = np.real(wfn_sim.expectation(pyquil_prog, pauli_terms=pyquil_obs_z))

E_final_qvm = np.concatenate([E_x_qvm, E_y_qvm, E_z_qvm])
print("QVM Evaluation Complete!\n")

def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

# Decode the bits directly from the QVM expectations
cur_bits = []
for i in range(num_nodes):
    if E_final_qvm[i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the initial base cut based on QVM
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"Initial Cut Size from PyQuil QVM before Local Search: {best_cut}")

# Single-bit swap search
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] 
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size (Post QVM):", best_cut)


Translating Qiskit circuit to PyQuil via PyTKET...
Translating observables...
Evaluating exact expectation values on PyQuil QVM WavefunctionSimulator...


/tmp/ipykernel_195921/3380019219.py:68: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=3)


QVM Evaluation Complete!

Initial Cut Size from PyQuil QVM before Local Search: 5834.7650154130315
Final Optimized Cut Size (Post QVM): 6507.965548513672


In [30]:
from pyquil import Program, get_qc
from pyquil.gates import H, RX, MEASURE
from pyquil.quil import address_qubits
import numpy as np

# ==========================================
# 9. APPLY MAPPING & EXECUTE ON ANKAA-3
# ==========================================
print("Preparing circuit for Rigetti Ankaa-3...")

# Your chosen high-fidelity patch
qubit_mapping = {
    0: 37, 1: 38, 2: 44, 3: 45, 4: 51, 5: 52, 
    6: 53, 7: 58, 8: 59, 9: 60, 10: 30, 11: 31
}

# 1. Map the logical PyTKET program to your physical Ankaa-3 qubits
mapped_base_prog = address_qubits(pyquil_prog, qubit_mapping)

qc = get_qc("Ankaa-3")
shots = 5000

def run_mapped_basis(mapped_prog, basis, mapping, qc, shots):
    """Appends basis transformations, compiles for Ankaa-3, and executes."""
    # Enforce our mapping starting position
    prog = Program(f'PRAGMA INITIAL_REWIRE "NAIVE"')
    prog += mapped_prog
    
    # Declare classical register to hold our 12 bits
    ro = prog.declare("ro", "BIT", len(mapping))
    
    # 1. APPLY ALL GATES FIRST
    # Apply basis rotations to the physical qubits
    for logical_q, physical_q in mapping.items():
        if basis == 'X':
            prog += H(physical_q)
        elif basis == 'Y':
            prog += RX(np.pi/2, physical_q)
            
    # 2. APPLY ALL MEASUREMENTS LAST
    # Store physical qubit measurement in the corresponding logical index
    for logical_q, physical_q in mapping.items():
        prog += MEASURE(physical_q, ro[logical_q])
        
    prog.wrap_in_numshots_loop(shots)
    
    print(f"Compiling {basis}-basis circuit...")
    exe = qc.compile(prog)
    
    print(f"Executing {basis}-basis circuit on Ankaa-3...")
    result = qc.run(exe)
    
    return result.get_register_map().get("ro")

# 2. Run the 3 global tasks on the physical hardware
res_x = run_mapped_basis(mapped_base_prog, 'X', qubit_mapping, qc, shots)
res_y = run_mapped_basis(mapped_base_prog, 'Y', qubit_mapping, qc, shots)
res_z = run_mapped_basis(mapped_base_prog, 'Z', qubit_mapping, qc, shots)

# ==========================================
# 10. DECODE BITSTRINGS & CALCULATE CUT
# ==========================================
def get_expectations(bitstrings, qiskit_observables, shots):
    """Translates raw 0s and 1s into expectation values for our specific observables."""
    spins = 1 - 2 * bitstrings 
    evs = []
    
    for obs in qiskit_observables:
        pauli_str = list(reversed(obs.paulis.to_labels()[0]))
        obs_val = np.ones(shots)
        
        for q, char in enumerate(pauli_str):
            if char != 'I':
                obs_val *= spins[:, q] 
                
        evs.append(np.mean(obs_val))
    return np.array(evs)

print("\nCalculating expectations from Ankaa-3 telemetry...")
# Because we mapped the readouts back to logical indices (0-11), 
# our original observables list works perfectly here!
E_x_qpu = get_expectations(res_x, observables_x, shots)
E_y_qpu = get_expectations(res_y, observables_y, shots)
E_z_qpu = get_expectations(res_z, observables_z, shots)

E_final_qpu = np.concatenate([E_x_qpu, E_y_qpu, E_z_qpu])
print("Ankaa-3 QPU Evaluation Complete!\n")

def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if E_final_qpu[i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"Initial Cut Size from physical Ankaa-3 before Local Search: {best_cut}")

for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] 
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size (Post Ankaa-3 execution):", best_cut)

Preparing circuit for Rigetti Ankaa-3...
Compiling X-basis circuit...
Executing X-basis circuit on Ankaa-3...
Compiling Y-basis circuit...
Executing Y-basis circuit on Ankaa-3...
Compiling Z-basis circuit...
Executing Z-basis circuit on Ankaa-3...

Calculating expectations from Ankaa-3 telemetry...
Ankaa-3 QPU Evaluation Complete!

Initial Cut Size from physical Ankaa-3 before Local Search: 2640.792568912367
Final Optimized Cut Size (Post Ankaa-3 execution): 6077.970355349812


In [36]:
from pytket.extensions.qiskit import qiskit_to_tk
from pytket.extensions.pyquil import tk_to_pyquil
from pyquil.paulis import sI, sX, sY, sZ
from pytket.passes import AutoRebase
from pyquil.api import WavefunctionSimulator
import numpy as np
from pytket.extensions.qiskit import qiskit_to_tk
from pytket.extensions.pyquil import tk_to_pyquil
from pytket.passes import AutoRebase
from pytket.circuit import OpType
from pyquil.paulis import sI, sX, sY, sZ
from pyquil.api import WavefunctionSimulator
import numpy as np

# ==========================================
# 8. TRANSLATE TO NATIVE PYQUIL VIA PYTKET
# ==========================================
print("\nTranslating Qiskit circuit to PyQuil via PyTKET...")

# 1. Bind the trained parameters to your Qiskit ansatz




df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 12                 # Increased to 12 to support 180 nodes at k=2
k = 2                           # Quadratic compression

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        # This loop makes it dynamically work for any k
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=2)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])








best_params_loaded = np.load("pce_optimal_theta_adam_reps4qbt12Continue2010_2.npy")
bound_circuit = ansatz.assign_parameters(best_params_loaded)

# 2. Convert Qiskit -> TKET -> PyQuil Program
tket_circ = qiskit_to_tk(bound_circuit)


rebase = AutoRebase({OpType.CX, OpType.Rx, OpType.Rz})
rebase.apply(tket_circ)

pyquil_prog = tk_to_pyquil(tket_circ)

# 3. Translate Qiskit SparsePauliOp observables to PyQuil PauliTerms
def qiskit_to_pyquil_pauli(qiskit_op):
    """
    Converts a Qiskit SparsePauliOp to a PyQuil PauliTerm.
    Note: Qiskit strings are read right-to-left for qubit 0 -> n.
    """
    pauli_str = qiskit_op.paulis.to_labels()[0]
    term = sI()  # Start with the Identity multiplier
    
    # Enumerate backwards to match Qiskit's endianness
    for qubit_idx, char in enumerate(reversed(pauli_str)):
        if char == 'X':
            term *= sX(qubit_idx)
        elif char == 'Y':
            term *= sY(qubit_idx)
        elif char == 'Z':
            term *= sZ(qubit_idx)
    return term

print("Translating observables...")
pyquil_obs_x = [qiskit_to_pyquil_pauli(op) for op in observables_x]
pyquil_obs_y = [qiskit_to_pyquil_pauli(op) for op in observables_y]
pyquil_obs_z = [qiskit_to_pyquil_pauli(op) for op in observables_z]

# ==========================================
# 9. EVALUATE ON PYQUIL QVM & DECODE
# ==========================================
print("Evaluating exact expectation values on PyQuil QVM WavefunctionSimulator...")

wfn_sim = WavefunctionSimulator()

# Calculate expectation values by passing the entire list of PauliTerms directly!
# This natively returns a numpy array, which is much faster than looping.
E_x_qvm = np.real(wfn_sim.expectation(pyquil_prog, pauli_terms=pyquil_obs_x))
E_y_qvm = np.real(wfn_sim.expectation(pyquil_prog, pauli_terms=pyquil_obs_y))
E_z_qvm = np.real(wfn_sim.expectation(pyquil_prog, pauli_terms=pyquil_obs_z))

E_final_qvm = np.concatenate([E_x_qvm, E_y_qvm, E_z_qvm])
print("QVM Evaluation Complete!\n")

def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

# Decode the bits directly from the QVM expectations
cur_bits = []
for i in range(num_nodes):
    if E_final_qvm[i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the initial base cut based on QVM
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"Initial Cut Size from PyQuil QVM before Local Search: {best_cut}")

# Single-bit swap search
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] 
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size (Post QVM):", best_cut)


Translating Qiskit circuit to PyQuil via PyTKET...
Translating observables...
Evaluating exact expectation values on PyQuil QVM WavefunctionSimulator...


/tmp/ipykernel_195921/1720680072.py:68: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=2)


QVM Evaluation Complete!

Initial Cut Size from PyQuil QVM before Local Search: 5980.854814009903
Final Optimized Cut Size (Post QVM): 6586.349862227867


In [37]:
from pyquil.quilbase import Gate

two_qubit_gates = [
    inst for inst in pyquil_prog
    if isinstance(inst, Gate) and len(inst.qubits) == 2
]

print("Number of 2-qubit gates:", len(two_qubit_gates))

Number of 2-qubit gates: 22


In [38]:
from pyquil import Program, get_qc
from pyquil.gates import H, RX, MEASURE
from pyquil.quil import address_qubits
import numpy as np

# ==========================================
# 9. APPLY MAPPING & EXECUTE ON ANKAA-3
# ==========================================
print("Preparing circuit for Rigetti Ankaa-3...")

# Your chosen high-fidelity patch
qubit_mapping = {
    0: 37, 1: 38, 2: 44, 3: 45, 4: 51, 5: 52, 
    6: 53, 7: 58, 8: 59, 9: 60, 10: 30, 11: 31
}

# 1. Map the logical PyTKET program to your physical Ankaa-3 qubits
mapped_base_prog = address_qubits(pyquil_prog, qubit_mapping)

qc = get_qc("Ankaa-3")
shots = 5000

def run_mapped_basis(mapped_prog, basis, mapping, qc, shots):
    """Appends basis transformations, compiles for Ankaa-3, and executes."""
    # Enforce our mapping starting position
    prog = Program(f'PRAGMA INITIAL_REWIRE "NAIVE"')
    prog += mapped_prog
    
    # Declare classical register to hold our 12 bits
    ro = prog.declare("ro", "BIT", len(mapping))
    
    # 1. APPLY ALL GATES FIRST
    # Apply basis rotations to the physical qubits
    for logical_q, physical_q in mapping.items():
        if basis == 'X':
            prog += H(physical_q)
        elif basis == 'Y':
            prog += RX(np.pi/2, physical_q)
            
    # 2. APPLY ALL MEASUREMENTS LAST
    # Store physical qubit measurement in the corresponding logical index
    for logical_q, physical_q in mapping.items():
        prog += MEASURE(physical_q, ro[logical_q])
        
    prog.wrap_in_numshots_loop(shots)
    
    print(f"Compiling {basis}-basis circuit...")
    exe = qc.compile(prog)
    
    print(f"Executing {basis}-basis circuit on Ankaa-3...")
    result = qc.run(exe)
    
    return result.get_register_map().get("ro")

# 2. Run the 3 global tasks on the physical hardware
res_x = run_mapped_basis(mapped_base_prog, 'X', qubit_mapping, qc, shots)
res_y = run_mapped_basis(mapped_base_prog, 'Y', qubit_mapping, qc, shots)
res_z = run_mapped_basis(mapped_base_prog, 'Z', qubit_mapping, qc, shots)

# ==========================================
# 10. DECODE BITSTRINGS & CALCULATE CUT
# ==========================================
def get_expectations(bitstrings, qiskit_observables, shots):
    """Translates raw 0s and 1s into expectation values for our specific observables."""
    spins = 1 - 2 * bitstrings 
    evs = []
    
    for obs in qiskit_observables:
        pauli_str = list(reversed(obs.paulis.to_labels()[0]))
        obs_val = np.ones(shots)
        
        for q, char in enumerate(pauli_str):
            if char != 'I':
                obs_val *= spins[:, q] 
                
        evs.append(np.mean(obs_val))
    return np.array(evs)

print("\nCalculating expectations from Ankaa-3 telemetry...")
# Because we mapped the readouts back to logical indices (0-11), 
# our original observables list works perfectly here!
E_x_qpu = get_expectations(res_x, observables_x, shots)
E_y_qpu = get_expectations(res_y, observables_y, shots)
E_z_qpu = get_expectations(res_z, observables_z, shots)

E_final_qpu = np.concatenate([E_x_qpu, E_y_qpu, E_z_qpu])
print("Ankaa-3 QPU Evaluation Complete!\n")

def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if E_final_qpu[i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"Initial Cut Size from physical Ankaa-3 before Local Search: {best_cut}")

for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] 
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size (Post Ankaa-3 execution):", best_cut)

Preparing circuit for Rigetti Ankaa-3...
Compiling X-basis circuit...
Executing X-basis circuit on Ankaa-3...
Compiling Y-basis circuit...
Executing Y-basis circuit on Ankaa-3...
Compiling Z-basis circuit...
Executing Z-basis circuit on Ankaa-3...

Calculating expectations from Ankaa-3 telemetry...
Ankaa-3 QPU Evaluation Complete!

Initial Cut Size from physical Ankaa-3 before Local Search: 3298.433232997242
Final Optimized Cut Size (Post Ankaa-3 execution): 5905.888565677614


In [39]:
from collections import Counter

def aggressive_cut_search(bitstrings, graph, top_k=50):
    counts = Counter(map(tuple, bitstrings))
    top_candidates = counts.most_common(top_k)

    best_cut = -1
    best_bits = None

    for bits, _ in top_candidates:
        partition = [set(), set()]
        for i, bit in enumerate(bits):
            partition[bit].add(i)

        cut = calc_cut_size(graph, partition[0], partition[1])

        if cut > best_cut:
            best_cut = cut
            best_bits = bits

    return best_cut, best_bits

best_cut_hist, best_bits_hist = aggressive_cut_search(res_z, graph, top_k=100)

print("Best Cut from histogram search:", best_cut_hist)

Best Cut from histogram search: 0


In [43]:
import numpy as np

# ==========================================
# 10. AGGRESSIVE POST-PROCESSING & CUT CALC
# ==========================================

def calc_cut_size(graph, bitstring):
    """Calculates cut size, dynamically handling graph node labels."""
    cut_size = 0
    node_list = list(graph.nodes())
    
    for edge0, edge1, data in graph.edges(data=True):
        # Find the 0-based index for each edge to match the bitstring
        idx0 = edge0 if isinstance(edge0, int) and edge0 < len(bitstring) else node_list.index(edge0)
        idx1 = edge1 if isinstance(edge1, int) and edge1 < len(bitstring) else node_list.index(edge1)
        
        w = data.get('weight', 1.0)
        if bitstring[idx0] != bitstring[idx1]:
            cut_size += w
            
    return cut_size

def aggressive_local_search(graph, initial_bits):
    """
    Runs a greedy local search to absolute convergence (local optima), 
    rather than stopping after just one pass.
    """
    best_bits = initial_bits.copy()
    best_cut = calc_cut_size(graph, best_bits)
    
    improved = True
    while improved:
        improved = False
        for node in range(len(best_bits)):
            # Flip the bit
            swapped_bits = best_bits.copy()
            swapped_bits[node] = 1 - swapped_bits[node]
            
            # Evaluate new cut
            cut_size = calc_cut_size(graph, swapped_bits)
            
            # If strictly better, keep it and flag that we improved
            if cut_size > best_cut:
                best_cut = cut_size
                best_bits = swapped_bits
                improved = True # Forces another full pass over the graph
                
    return best_bits, best_cut

print("\nExecuting Aggressive Post-Processing on Ankaa-3 telemetry...")

# 1. Generate the initial state using YOUR combined X, Y, Z encoding
cur_bits = []
for i in range(num_nodes):
    if E_final_qpu[i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the starting cut size based on QPU telemetry
initial_qpu_cut = calc_cut_size(graph, cur_bits)
print(f"Initial Cut Size from physical Ankaa-3 before Local Search: {initial_qpu_cut}")

# 2. Run the aggressive converging local search on this candidate
optimized_bits, optimized_cut = aggressive_local_search(graph, cur_bits)

print("--- Post-Processing Results ---")
print(f"Final Optimized Cut Size (Aggressive Search):   {optimized_cut}")
# print(f"Winning Node Partitioning: {optimized_bits}") # Uncomment to see the array


Executing Aggressive Post-Processing on Ankaa-3 telemetry...
Initial Cut Size from physical Ankaa-3 before Local Search: 3298.433232997242
--- Post-Processing Results ---
Final Optimized Cut Size (Aggressive Search):   6489.682193213306


In [44]:
from pyquil import Program, get_qc
from pyquil.gates import H, RX, MEASURE
from pyquil.quil import address_qubits
import numpy as np

# ==========================================
# 9. APPLY MAPPING & EXECUTE ON ANKAA-3
# ==========================================
print("Preparing circuit for Rigetti Ankaa-3...")

# Your chosen high-fidelity patch
qubit_mapping = {
    0: 37, 1: 38, 2: 44, 3: 45, 4: 51, 5: 52, 
    6: 53, 7: 58, 8: 59, 9: 60, 10: 30, 11: 31
}

# 1. Map the logical PyTKET program to your physical Ankaa-3 qubits
mapped_base_prog = address_qubits(pyquil_prog, qubit_mapping)

qc = get_qc("Ankaa-3")
shots = 10000

def run_mapped_basis(mapped_prog, basis, mapping, qc, shots):
    """Appends basis transformations, compiles for Ankaa-3, and executes."""
    # Enforce our mapping starting position
    prog = Program(f'PRAGMA INITIAL_REWIRE "NAIVE"')
    prog += mapped_prog
    
    # Declare classical register to hold our 12 bits
    ro = prog.declare("ro", "BIT", len(mapping))
    
    # 1. APPLY ALL GATES FIRST
    # Apply basis rotations to the physical qubits
    for logical_q, physical_q in mapping.items():
        if basis == 'X':
            prog += H(physical_q)
        elif basis == 'Y':
            prog += RX(np.pi/2, physical_q)
            
    # 2. APPLY ALL MEASUREMENTS LAST
    # Store physical qubit measurement in the corresponding logical index
    for logical_q, physical_q in mapping.items():
        prog += MEASURE(physical_q, ro[logical_q])
        
    prog.wrap_in_numshots_loop(shots)
    
    print(f"Compiling {basis}-basis circuit...")
    exe = qc.compile(prog)
    
    print(f"Executing {basis}-basis circuit on Ankaa-3...")
    result = qc.run(exe)
    
    return result.get_register_map().get("ro")

# 2. Run the 3 global tasks on the physical hardware
res_x = run_mapped_basis(mapped_base_prog, 'X', qubit_mapping, qc, shots)
res_y = run_mapped_basis(mapped_base_prog, 'Y', qubit_mapping, qc, shots)
res_z = run_mapped_basis(mapped_base_prog, 'Z', qubit_mapping, qc, shots)

# ==========================================
# 10. DECODE BITSTRINGS & CALCULATE CUT
# ==========================================
def get_expectations(bitstrings, qiskit_observables, shots):
    """Translates raw 0s and 1s into expectation values for our specific observables."""
    spins = 1 - 2 * bitstrings 
    evs = []
    
    for obs in qiskit_observables:
        pauli_str = list(reversed(obs.paulis.to_labels()[0]))
        obs_val = np.ones(shots)
        
        for q, char in enumerate(pauli_str):
            if char != 'I':
                obs_val *= spins[:, q] 
                
        evs.append(np.mean(obs_val))
    return np.array(evs)

print("\nCalculating expectations from Ankaa-3 telemetry...")
# Because we mapped the readouts back to logical indices (0-11), 
# our original observables list works perfectly here!
E_x_qpu = get_expectations(res_x, observables_x, shots)
E_y_qpu = get_expectations(res_y, observables_y, shots)
E_z_qpu = get_expectations(res_z, observables_z, shots)

E_final_qpu = np.concatenate([E_x_qpu, E_y_qpu, E_z_qpu])
print("Ankaa-3 QPU Evaluation Complete!\n")

def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if E_final_qpu[i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"Initial Cut Size from physical Ankaa-3 before Local Search: {best_cut}")

for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] 
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size (Post Ankaa-3 execution):", best_cut)

Preparing circuit for Rigetti Ankaa-3...
Compiling X-basis circuit...
Executing X-basis circuit on Ankaa-3...
Compiling Y-basis circuit...
Executing Y-basis circuit on Ankaa-3...
Compiling Z-basis circuit...
Executing Z-basis circuit on Ankaa-3...

Calculating expectations from Ankaa-3 telemetry...
Ankaa-3 QPU Evaluation Complete!

Initial Cut Size from physical Ankaa-3 before Local Search: 3285.1695232678703
Final Optimized Cut Size (Post Ankaa-3 execution): 6109.015154711543


In [45]:
import numpy as np

# ==========================================
# 10. AGGRESSIVE POST-PROCESSING & CUT CALC
# ==========================================

def calc_cut_size(graph, bitstring):
    """Calculates cut size, dynamically handling graph node labels."""
    cut_size = 0
    node_list = list(graph.nodes())
    
    for edge0, edge1, data in graph.edges(data=True):
        # Find the 0-based index for each edge to match the bitstring
        idx0 = edge0 if isinstance(edge0, int) and edge0 < len(bitstring) else node_list.index(edge0)
        idx1 = edge1 if isinstance(edge1, int) and edge1 < len(bitstring) else node_list.index(edge1)
        
        w = data.get('weight', 1.0)
        if bitstring[idx0] != bitstring[idx1]:
            cut_size += w
            
    return cut_size

def aggressive_local_search(graph, initial_bits):
    """
    Runs a greedy local search to absolute convergence (local optima), 
    rather than stopping after just one pass.
    """
    best_bits = initial_bits.copy()
    best_cut = calc_cut_size(graph, best_bits)
    
    improved = True
    while improved:
        improved = False
        for node in range(len(best_bits)):
            # Flip the bit
            swapped_bits = best_bits.copy()
            swapped_bits[node] = 1 - swapped_bits[node]
            
            # Evaluate new cut
            cut_size = calc_cut_size(graph, swapped_bits)
            
            # If strictly better, keep it and flag that we improved
            if cut_size > best_cut:
                best_cut = cut_size
                best_bits = swapped_bits
                improved = True # Forces another full pass over the graph
                
    return best_bits, best_cut

print("\nExecuting Aggressive Post-Processing on Ankaa-3 telemetry...")

# 1. Generate the initial state using YOUR combined X, Y, Z encoding
cur_bits = []
for i in range(num_nodes):
    if E_final_qpu[i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the starting cut size based on QPU telemetry
initial_qpu_cut = calc_cut_size(graph, cur_bits)
print(f"Initial Cut Size from physical Ankaa-3 before Local Search: {initial_qpu_cut}")

# 2. Run the aggressive converging local search on this candidate
optimized_bits, optimized_cut = aggressive_local_search(graph, cur_bits)

print("--- Post-Processing Results ---")
print(f"Final Optimized Cut Size (Aggressive Search):   {optimized_cut}")
# print(f"Winning Node Partitioning: {optimized_bits}") # Uncomment to see the array


Executing Aggressive Post-Processing on Ankaa-3 telemetry...
Initial Cut Size from physical Ankaa-3 before Local Search: 3285.1695232678703
--- Post-Processing Results ---
Final Optimized Cut Size (Aggressive Search):   6482.379762539362


In [46]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile
from qiskit.primitives import BackendEstimatorV2


# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 12                 # Increased to 12 to support 180 nodes at k=2
k = 2                           # Quadratic compression

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        # This loop makes it dynamically work for any k
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
# High shots reduce statistical noise for the parameter shift gradients
estimator.options.default_shots = 10000

experiment_result = []


from pytket.extensions.qiskit import qiskit_to_tk
from pytket.extensions.pyquil import tk_to_pyquil
from pyquil.paulis import sI, sX, sY, sZ
from pytket.passes import AutoRebase
from pyquil.api import WavefunctionSimulator
import numpy as np
from pytket.extensions.qiskit import qiskit_to_tk
from pytket.extensions.pyquil import tk_to_pyquil
from pytket.passes import AutoRebase
from pytket.circuit import OpType
from pyquil.paulis import sI, sX, sY, sZ
from pyquil.api import WavefunctionSimulator
import numpy as np

# ==========================================
# 8. TRANSLATE TO NATIVE PYQUIL VIA PYTKET
# ==========================================
print("\nTranslating Qiskit circuit to PyQuil via PyTKET...")

# 1. Bind the trained parameters to your Qiskit ansatz
best_params_loaded = np.load("pce_optimal_theta_adam_reps4qbt12Continue2004.npy")
bound_circuit = ansatz.assign_parameters(best_params_loaded)

# 2. Convert Qiskit -> TKET -> PyQuil Program
tket_circ = qiskit_to_tk(bound_circuit)


rebase = AutoRebase({OpType.CX, OpType.Rx, OpType.Rz})
rebase.apply(tket_circ)

pyquil_prog = tk_to_pyquil(tket_circ)

# 3. Translate Qiskit SparsePauliOp observables to PyQuil PauliTerms
def qiskit_to_pyquil_pauli(qiskit_op):
    """
    Converts a Qiskit SparsePauliOp to a PyQuil PauliTerm.
    Note: Qiskit strings are read right-to-left for qubit 0 -> n.
    """
    pauli_str = qiskit_op.paulis.to_labels()[0]
    term = sI()  # Start with the Identity multiplier
    
    # Enumerate backwards to match Qiskit's endianness
    for qubit_idx, char in enumerate(reversed(pauli_str)):
        if char == 'X':
            term *= sX(qubit_idx)
        elif char == 'Y':
            term *= sY(qubit_idx)
        elif char == 'Z':
            term *= sZ(qubit_idx)
    return term

print("Translating observables...")
pyquil_obs_x = [qiskit_to_pyquil_pauli(op) for op in observables_x]
pyquil_obs_y = [qiskit_to_pyquil_pauli(op) for op in observables_y]
pyquil_obs_z = [qiskit_to_pyquil_pauli(op) for op in observables_z]

# ==========================================
# 9. EVALUATE ON PYQUIL QVM & DECODE
# ==========================================
print("Evaluating exact expectation values on PyQuil QVM WavefunctionSimulator...")

wfn_sim = WavefunctionSimulator()

# Calculate expectation values by passing the entire list of PauliTerms directly!
# This natively returns a numpy array, which is much faster than looping.
E_x_qvm = np.real(wfn_sim.expectation(pyquil_prog, pauli_terms=pyquil_obs_x))
E_y_qvm = np.real(wfn_sim.expectation(pyquil_prog, pauli_terms=pyquil_obs_y))
E_z_qvm = np.real(wfn_sim.expectation(pyquil_prog, pauli_terms=pyquil_obs_z))

E_final_qvm = np.concatenate([E_x_qvm, E_y_qvm, E_z_qvm])
print("QVM Evaluation Complete!\n")

def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

# Decode the bits directly from the QVM expectations
cur_bits = []
for i in range(num_nodes):
    if E_final_qvm[i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the initial base cut based on QVM
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"Initial Cut Size from PyQuil QVM before Local Search: {best_cut}")

# Single-bit swap search
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] 
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size (Post QVM):", best_cut)


from pyquil.quilbase import Gate

two_qubit_gates = [
    inst for inst in pyquil_prog
    if isinstance(inst, Gate) and len(inst.qubits) == 2
]

print("Number of 2-qubit gates:", len(two_qubit_gates))

Loading dataset...

Translating Qiskit circuit to PyQuil via PyTKET...
Translating observables...
Evaluating exact expectation values on PyQuil QVM WavefunctionSimulator...


/tmp/ipykernel_195921/3139097806.py:61: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)


QVM Evaluation Complete!

Initial Cut Size from PyQuil QVM before Local Search: 6260.286774176787
Final Optimized Cut Size (Post QVM): 6528.332178117385
Number of 2-qubit gates: 44


In [47]:
from pyquil import Program, get_qc
from pyquil.gates import H, RX, MEASURE
from pyquil.quil import address_qubits
import numpy as np

# ==========================================
# 9. APPLY MAPPING & EXECUTE ON ANKAA-3
# ==========================================
print("Preparing circuit for Rigetti Ankaa-3...")

# Your chosen high-fidelity patch
qubit_mapping = {
    0: 37, 1: 38, 2: 44, 3: 45, 4: 51, 5: 52, 
    6: 53, 7: 58, 8: 59, 9: 60, 10: 30, 11: 31
}

# 1. Map the logical PyTKET program to your physical Ankaa-3 qubits
mapped_base_prog = address_qubits(pyquil_prog, qubit_mapping)

qc = get_qc("Ankaa-3")
shots = 5000

def run_mapped_basis(mapped_prog, basis, mapping, qc, shots):
    """Appends basis transformations, compiles for Ankaa-3, and executes."""
    # Enforce our mapping starting position
    prog = Program(f'PRAGMA INITIAL_REWIRE "NAIVE"')
    prog += mapped_prog
    
    # Declare classical register to hold our 12 bits
    ro = prog.declare("ro", "BIT", len(mapping))
    
    # 1. APPLY ALL GATES FIRST
    # Apply basis rotations to the physical qubits
    for logical_q, physical_q in mapping.items():
        if basis == 'X':
            prog += H(physical_q)
        elif basis == 'Y':
            prog += RX(np.pi/2, physical_q)
            
    # 2. APPLY ALL MEASUREMENTS LAST
    # Store physical qubit measurement in the corresponding logical index
    for logical_q, physical_q in mapping.items():
        prog += MEASURE(physical_q, ro[logical_q])
        
    prog.wrap_in_numshots_loop(shots)
    
    print(f"Compiling {basis}-basis circuit...")
    exe = qc.compile(prog)
    
    print(f"Executing {basis}-basis circuit on Ankaa-3...")
    result = qc.run(exe)
    
    return result.get_register_map().get("ro")

# 2. Run the 3 global tasks on the physical hardware
res_x = run_mapped_basis(mapped_base_prog, 'X', qubit_mapping, qc, shots)
res_y = run_mapped_basis(mapped_base_prog, 'Y', qubit_mapping, qc, shots)
res_z = run_mapped_basis(mapped_base_prog, 'Z', qubit_mapping, qc, shots)

# ==========================================
# 10. DECODE BITSTRINGS & CALCULATE CUT
# ==========================================
def get_expectations(bitstrings, qiskit_observables, shots):
    """Translates raw 0s and 1s into expectation values for our specific observables."""
    spins = 1 - 2 * bitstrings 
    evs = []
    
    for obs in qiskit_observables:
        pauli_str = list(reversed(obs.paulis.to_labels()[0]))
        obs_val = np.ones(shots)
        
        for q, char in enumerate(pauli_str):
            if char != 'I':
                obs_val *= spins[:, q] 
                
        evs.append(np.mean(obs_val))
    return np.array(evs)

print("\nCalculating expectations from Ankaa-3 telemetry...")
# Because we mapped the readouts back to logical indices (0-11), 
# our original observables list works perfectly here!
E_x_qpu = get_expectations(res_x, observables_x, shots)
E_y_qpu = get_expectations(res_y, observables_y, shots)
E_z_qpu = get_expectations(res_z, observables_z, shots)

E_final_qpu = np.concatenate([E_x_qpu, E_y_qpu, E_z_qpu])
print("Ankaa-3 QPU Evaluation Complete!\n")

def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if E_final_qpu[i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"Initial Cut Size from physical Ankaa-3 before Local Search: {best_cut}")

for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] 
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size (Post Ankaa-3 execution):", best_cut)

Preparing circuit for Rigetti Ankaa-3...
Compiling X-basis circuit...
Executing X-basis circuit on Ankaa-3...
Compiling Y-basis circuit...
Executing Y-basis circuit on Ankaa-3...
Compiling Z-basis circuit...
Executing Z-basis circuit on Ankaa-3...

Calculating expectations from Ankaa-3 telemetry...
Ankaa-3 QPU Evaluation Complete!

Initial Cut Size from physical Ankaa-3 before Local Search: 3123.9517257274506
Final Optimized Cut Size (Post Ankaa-3 execution): 6239.70292558976


In [48]:
import numpy as np

# ==========================================
# 10. AGGRESSIVE POST-PROCESSING & CUT CALC
# ==========================================

def calc_cut_size(graph, bitstring):
    """Calculates cut size, dynamically handling graph node labels."""
    cut_size = 0
    node_list = list(graph.nodes())
    
    for edge0, edge1, data in graph.edges(data=True):
        # Find the 0-based index for each edge to match the bitstring
        idx0 = edge0 if isinstance(edge0, int) and edge0 < len(bitstring) else node_list.index(edge0)
        idx1 = edge1 if isinstance(edge1, int) and edge1 < len(bitstring) else node_list.index(edge1)
        
        w = data.get('weight', 1.0)
        if bitstring[idx0] != bitstring[idx1]:
            cut_size += w
            
    return cut_size

def aggressive_local_search(graph, initial_bits):
    """
    Runs a greedy local search to absolute convergence (local optima), 
    rather than stopping after just one pass.
    """
    best_bits = initial_bits.copy()
    best_cut = calc_cut_size(graph, best_bits)
    
    improved = True
    while improved:
        improved = False
        for node in range(len(best_bits)):
            # Flip the bit
            swapped_bits = best_bits.copy()
            swapped_bits[node] = 1 - swapped_bits[node]
            
            # Evaluate new cut
            cut_size = calc_cut_size(graph, swapped_bits)
            
            # If strictly better, keep it and flag that we improved
            if cut_size > best_cut:
                best_cut = cut_size
                best_bits = swapped_bits
                improved = True # Forces another full pass over the graph
                
    return best_bits, best_cut

print("\nExecuting Aggressive Post-Processing on Ankaa-3 telemetry...")

# 1. Generate the initial state using YOUR combined X, Y, Z encoding
cur_bits = []
for i in range(num_nodes):
    if E_final_qpu[i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the starting cut size based on QPU telemetry
initial_qpu_cut = calc_cut_size(graph, cur_bits)
print(f"Initial Cut Size from physical Ankaa-3 before Local Search: {initial_qpu_cut}")

# 2. Run the aggressive converging local search on this candidate
optimized_bits, optimized_cut = aggressive_local_search(graph, cur_bits)

print("--- Post-Processing Results ---")
print(f"Final Optimized Cut Size (Aggressive Search):   {optimized_cut}")
# print(f"Winning Node Partitioning: {optimized_bits}") # Uncomment to see the array


Executing Aggressive Post-Processing on Ankaa-3 telemetry...
Initial Cut Size from physical Ankaa-3 before Local Search: 3123.9517257274506
--- Post-Processing Results ---
Final Optimized Cut Size (Aggressive Search):   6593.444988213511


In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# Import SPSA from qiskit algorithms
from qiskit_algorithms.optimizers import SPSA

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 12                 # Increased to 12 to support 180 nodes at k=2
k = 2                           # Quadratic compression

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=2)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
estimator.options.default_shots = 10000

# ==========================================
# 4. OBJECTIVE LOSS FUNCTION
# ==========================================
alpha_loss = 24  
beta_loss = 0.5
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def loss_func(theta):
    """Clean scalar objective function for SPSA."""
    job = estimator.run([
        (ansatz, observables_x, theta),
        (ansatz, observables_y, theta),
        (ansatz, observables_z, theta)
    ])
    results = job.result()
    
    # Extract 1D arrays of expectation values
    E_x = results[0].data.evs
    E_y = results[1].data.evs
    E_z = results[2].data.evs
    E = np.concatenate([E_x, E_y, E_z])
    
    tE = np.tanh(alpha_loss * E)
    
    loss_cut = 0.0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss_cut += w * tE[edge0] * tE[edge1]
        
    S = np.sum(tE**2) / num_nodes
    reg_term = beta_loss * v * (S**2)
    
    return loss_cut + reg_term

# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue2010.npy"
num_new_params = ansatz.num_parameters 

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        initial_params = old_params
    elif num_old_params < num_new_params:
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        initial_params[:num_old_params] = old_params
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)

# ==========================================
# 6. SPSA OPTIMIZER
# ==========================================
print("\nStarting SPSA Optimization...")

# Track the absolute best parameters seen by SPSA
best_loss = float('inf')
best_params = np.copy(initial_params)

def spsa_callback(nfev, x_next, fx_next, stepsize, accepted):
    """Callback to print progress and track the best overall parameters."""
    global best_loss, best_params
    step = int(nfev / 2)
    
    if fx_next < best_loss:
        best_loss = fx_next
        best_params = np.copy(x_next)
        
    print(f"--- SPSA Step {step} --- | Current Loss: {fx_next:.6f} | Best Loss: {best_loss:.6f}")

# Initialize SPSA. Adjust maxiter to control how long it runs.
spsa = SPSA(maxiter=500, callback=spsa_callback)

# Run the minimization
result = spsa.minimize(fun=loss_func, x0=initial_params)

print("\nSPSA Optimization Complete!")
np.save("pce_optimal_theta_spsa_reps2qbt12_linear_spsa.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

print("Running one final evaluation to extract exact expectation values...")
# Run the estimator once with the absolute best parameters found
final_job = estimator.run([
    (ansatz, observables_x, best_params),
    (ansatz, observables_y, best_params),
    (ansatz, observables_z, best_params)
])
final_results = final_job.result()
final_E = np.concatenate([
    final_results[0].data.evs,
    final_results[1].data.evs,
    final_results[2].data.evs
])

# Thresholding into classical bits
cur_bits = []
for i in range(num_nodes):
    if final_E[i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the initial base cut
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"\nInitial Cut Size before Local Search: {best_cut}")

# Single-bit swap search as described in the paper
print("Running greedy 1-bit swap search...")
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] # Flip 1 to 0 or 0 to 1
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    # Keep the swap if it improves the cut
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("=====================================")
print(f"🎯 Final Optimized Cut Size: {best_cut}")
print("=====================================")

Loading dataset...


/tmp/ipykernel_200568/996380004.py:61: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=2)


Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue2010.npy...

Starting SPSA Optimization...
--- SPSA Step 1 --- | Current Loss: -267.146795 | Best Loss: -267.146795
--- SPSA Step 3 --- | Current Loss: -352.646000 | Best Loss: -352.646000
--- SPSA Step 4 --- | Current Loss: -312.434561 | Best Loss: -352.646000
--- SPSA Step 6 --- | Current Loss: -165.792724 | Best Loss: -352.646000
--- SPSA Step 7 --- | Current Loss: 97.545694 | Best Loss: -352.646000
--- SPSA Step 9 --- | Current Loss: -75.505965 | Best Loss: -352.646000
--- SPSA Step 10 --- | Current Loss: -85.371810 | Best Loss: -352.646000
--- SPSA Step 12 --- | Current Loss: -115.163341 | Best Loss: -352.646000
--- SPSA Step 13 --- | Current Loss: -114.926627 | Best Loss: -352.646000
--- SPSA Step 15 --- | Current Loss: -106.604785 | Best Loss: -352.646000
--- SPSA Step 16 --- | Current Loss: -57.506683 | Best Loss: -352.646000
--- SPSA Step 18 --- | Current Loss: -24.815079 | Best Loss: -352.646000
--- S

In [2]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 12                 # Increased to 12 to support 180 nodes at k=2
k = 2                           # Quadratic compression

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        # This loop makes it dynamically work for any k
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=3)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
# High shots reduce statistical noise for the parameter shift gradients
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. LOSS FUNCTION & CHAIN RULE GRADIENTS
# ==========================================
alpha_loss = 24  # Rescaling factor scaled to approx n^(k/2)
beta_loss = 0.5
# Edwards-Erdös bound for the regularization parameter v
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def evaluate_expectations(theta):
    """Helper to get pure expectation values for a single parameter array."""
    job = estimator.run([
        (ansatz, observables_x, theta),
        (ansatz, observables_y, theta),
        (ansatz, observables_z, theta)
    ])
    results = job.result()
    
    # Extract 1D arrays of expectation values
    E_x = results[0].data.evs
    E_y = results[1].data.evs
    E_z = results[2].data.evs
    return np.concatenate([E_x, E_y, E_z])

def calc_loss_and_chain_rule(E):
    """Analytically calculates the loss and the gradient dL/dE."""
    # Precompute common terms for speed
    tE = np.tanh(alpha_loss * E)
    sech2E = 1.0 - tE**2
    
    loss_cut = 0.0
    dL_dE = np.zeros(num_nodes)
    
    # Chain rule for the Cut Loss term
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss_cut += w * tE[edge0] * tE[edge1]
        
        # dL/dE for connected nodes
        dL_dE[edge0] += w * alpha_loss * sech2E[edge0] * tE[edge1]
        dL_dE[edge1] += w * alpha_loss * sech2E[edge1] * tE[edge0]
        
    # Chain rule for the Regularization term
    S = np.sum(tE**2) / num_nodes
    reg_term = beta_loss * v * (S**2)
    loss = loss_cut + reg_term
    
    # Add regularization gradient to dL/dE
    reg_grad_factor = (4.0 * beta_loss * v * S / num_nodes) * alpha_loss
    dL_dE += reg_grad_factor * tE * sech2E
    
    return loss, dL_dE

# ==========================================
# 4.5. PARAMETER SHIFT RULE JACOBIAN (CORRECTED)
# ==========================================
def param_shift_gradient(theta):
    """Calculates the full gradient using batched Parameter Shift + Chain Rule."""
    num_params = len(theta)
    shift = np.pi / 2.0
    
    # Step A: Build a batched array of shifted parameters
    theta_shifted = []
    for j in range(num_params):
        t_plus = theta.copy()
        t_plus[j] += shift
        t_minus = theta.copy()
        t_minus[j] -= shift
        theta_shifted.append(t_plus)
        theta_shifted.append(t_minus)
        
    theta_shifted = np.array(theta_shifted) # Shape: (2*num_params, num_params)
    
    # FIX: Add a new axis to force a Cartesian product broadcast
    # Now it evaluates all 240 parameter sets against all 60 observables
    theta_shifted_broadcast = theta_shifted[:, np.newaxis, :] 
    
    # Step B: Run batched estimator
    job = estimator.run([
        (ansatz, observables_x, theta_shifted_broadcast),
        (ansatz, observables_y, theta_shifted_broadcast),
        (ansatz, observables_z, theta_shifted_broadcast)
    ])
    results = job.result()
    
    # Safely extract and orient the results to shape (len(observables), 2*num_params)
    def extract_evs(pub_result):
        evs = np.atleast_2d(pub_result.data.evs)
        # Transpose if the shape is (240, 60) to make it (60, 240)
        if evs.shape[0] == 2 * num_params:
            evs = evs.T
        return evs

    evs_x = extract_evs(results[0])
    evs_y = extract_evs(results[1])
    evs_z = extract_evs(results[2])
    
    # Stack vertically to get shape (num_nodes, 2*num_params)
    all_evs = np.vstack([evs_x, evs_y, evs_z])
    
    # Step C: Compute Jacobian dE/dTheta
    J = np.zeros((num_nodes, num_params))
    for j in range(num_params):
        evs_plus = all_evs[:, 2*j]
        evs_minus = all_evs[:, 2*j + 1]
        J[:, j] = (evs_plus - evs_minus) / 2.0
        
    # Step D: Apply Chain Rule
    E_current = evaluate_expectations(theta)
    loss, dL_dE = calc_loss_and_chain_rule(E_current)
    grad = dL_dE @ J
    
    # Save for bit-swap decoding
    global experiment_result
    experiment_result.append({"loss": loss, "exp_map": dict(enumerate(E_current))})
    
    return loss, grad

# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue2010.npy"
num_new_params = ansatz.num_parameters # For reps=4 and 12 qubits, this will be 120

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        initial_params = old_params
    elif num_old_params < num_new_params:
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        initial_params[:num_old_params] = old_params
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)

# ==========================================
# 6. CUSTOM ADAM OPTIMIZER
# ==========================================
def adam_optimize_param_shift(theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=50):
    print(f"Starting optimization with Parameter Shift Adam (lr={lr})...")
    theta_i = np.array(theta0)
    
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    for i in range(max_iter):
        t += 1
        
        loss, grad = param_shift_gradient(theta_i)
        print(f"--- Adam Step {i+1}/{max_iter} --- | Loss: {loss:.6f}")
        
        m_t = beta1 * m_t + (1 - beta1) * grad
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        theta_next = theta_i - lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            break
            
        theta_i = theta_next
        
    return theta_i

best_params = adam_optimize_param_shift(
    theta0=initial_params,
    lr=0.2,        
    max_iter=1000
)

print("\nAdam Optimization Complete!")
np.save("pce_optimal_theta_adam_reps4qbt12Continue2010_linear3qbts.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the initial base cut
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"\nInitial Cut Size before Local Search: {best_cut}")

# Single-bit swap search as described in the paper
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] # Flip 1 to 0 or 0 to 1
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    # Keep the swap if it improves the cut
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue2010.npy...
Padding parameters! Transferring 72 angles into a 96 angle circuit.
Starting optimization with Parameter Shift Adam (lr=0.2)...


/tmp/ipykernel_200568/1642379906.py:59: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=3)


--- Adam Step 1/1000 --- | Loss: -177.059783
--- Adam Step 2/1000 --- | Loss: -602.494565
--- Adam Step 3/1000 --- | Loss: -926.840492
--- Adam Step 4/1000 --- | Loss: -1240.148885
--- Adam Step 5/1000 --- | Loss: -1572.897670
--- Adam Step 6/1000 --- | Loss: -1917.646147
--- Adam Step 7/1000 --- | Loss: -2186.056156
--- Adam Step 8/1000 --- | Loss: -2469.687450
--- Adam Step 9/1000 --- | Loss: -2492.412053
--- Adam Step 10/1000 --- | Loss: -2691.287693
--- Adam Step 11/1000 --- | Loss: -2953.855000
--- Adam Step 12/1000 --- | Loss: -3134.747697
--- Adam Step 13/1000 --- | Loss: -3167.397648
--- Adam Step 14/1000 --- | Loss: -3180.221545
--- Adam Step 15/1000 --- | Loss: -3187.972917
--- Adam Step 16/1000 --- | Loss: -3277.207637
--- Adam Step 17/1000 --- | Loss: -3290.461532
--- Adam Step 18/1000 --- | Loss: -3288.818526
--- Adam Step 19/1000 --- | Loss: -3321.757602
--- Adam Step 20/1000 --- | Loss: -3379.722778
--- Adam Step 21/1000 --- | Loss: -3431.036390
--- Adam Step 22/1000 ---

In [3]:
import networkx as nx

def find_best_chain_greedy(topology, qc, chain_length=12):
    """Finds a chain using a simple Greedy approach (always pick the best next neighbor)."""
    best_overall_chain = []
    best_overall_score = -1.0
    
    # Try starting the chain from every single qubit in the topology
    for start_node in topology.nodes:
        current_chain = [start_node]
        current_score = 1.0
        current_node = start_node
        
        # Build the chain step-by-step
        for _ in range(chain_length - 1):
            best_neighbor = None
            best_edge_score = -1.0
            
            # Look at all neighbors of the current qubit
            for neighbor in topology.neighbors(current_node):
                if neighbor not in current_chain:  # Don't loop back on ourselves
                    score = get_2q_fidelity(qc, current_node, neighbor)
                    
                    # Keep track of the highest fidelity neighbor
                    if score > best_edge_score:
                        best_edge_score = score
                        best_neighbor = neighbor
                        
            # If we hit a dead end (no unvisited neighbors), stop building this chain
            if best_neighbor is None:
                break
                
            # Move to the best neighbor and update the score
            current_chain.append(best_neighbor)
            current_score *= best_edge_score
            current_node = best_neighbor
            
        # If we successfully built a chain of length 12, check if it's our new global best
        if len(current_chain) == chain_length and current_score > best_overall_score:
            best_overall_score = current_score
            best_overall_chain = current_chain
            
    return best_overall_chain, best_overall_score

# ==========================================
# EXECUTE GREEDY SEARCH
# ==========================================
qc, topology = get_ankaa_topology()

if qc is None:
    topology = nx.convert_node_labels_to_integers(topology)

best_line, expected_fidelity = find_best_chain_greedy(topology, qc, chain_length=12)

print("\n" + "="*40)
print("🏆 OPTIMAL MAPPING FOUND (GREEDY)")
print("="*40)
print(f"Best Physical Qubit Chain: {best_line}")
print(f"Estimated Chain Fidelity: {expected_fidelity:.4f}")
print("\nMapping Dictionary:")
mapping_dict = {logical: physical for logical, physical in enumerate(best_line)}
print(mapping_dict)

NameError: name 'get_ankaa_topology' is not defined

In [ ]:
import networkx as nx
import random

def get_ankaa_topology():
    """
    Generates the topology for the Rigetti Ankaa processor (a 7x12 square lattice).
    Returns a mock backend object (dict) and the NetworkX graph.
    """
    # Create the 84-qubit square lattice (7 rows by 12 columns)
    topology = nx.grid_2d_graph(7, 12)
    
    # Convert the (row, col) coordinates to standard integer labels (0 to 83)
    topology = nx.convert_node_labels_to_integers(topology)
    
    # Create a mock 'qc' object holding edge fidelities
    # (If using real hardware later, replace this with your actual backend call)
    mock_qc = {}
    random.seed(42) # Keeps the "random" fidelities consistent every time you run it
    
    for u, v in topology.edges():
        # Simulate realistic two-qubit gate fidelities (e.g., 95% to 99.8%)
        fid = random.uniform(0.950, 0.998)
        mock_qc[(u, v)] = fid
        mock_qc[(v, u)] = fid # Ensure the edge is bidirectional
        
    return mock_qc, topology

def get_2q_fidelity(qc, q1, q2):
    """
    Helper function to safely extract the fidelity between two qubits.
    """
    # If the edge doesn't exist, return 0.0 (impossible to cross)
    return qc.get((q1, q2), 0.0)